# 00 — Getting Started: the joint spec and its canonical names

`midas_joint_ff_calibrate` performs **joint powder + FF-HEDM
differentiable calibration**.  Its job is to wire three existing
differentiable packages together over a single
`midas_peakfit.ParameterSpec`:

* `midas_calibrate_v2.loss.pseudo_strain` — the powder-calibrant residual
* `midas_fit_grain` / `midas_diffract` — the FF-HEDM spot residual
* `midas_peakfit` — spec / pack / LM / Laplace / Σ=0 gauge

This notebook builds the unified spec with `build_joint_spec`, prints
the **canonical parameter naming**, and shows the refined / frozen
split.  Self-contained synthetic data; runs in seconds.


In [1]:
import os, math, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

import midas_peakfit as mp
from midas_calibrate_v2.forward.panels import PanelLayout
from midas_hkls import Lattice, SpaceGroup
from midas_diffract.hkls import hkls_for_forward_model

# The package's validated synthetic generators (paper fig. 1 source).
import midas_joint_ff_calibrate.runners.run_synthetic_pilatus_joint as R
from midas_joint_ff_calibrate.loss import JointWeights, joint_residual
from midas_joint_ff_calibrate.pipelines.identifiability import fisher_block_rank
from midas_joint_ff_calibrate.pipelines.alternating import AlternatingDriver
from midas_joint_ff_calibrate.pipelines.full_joint import FullJointDriver

# ---- shrink the problem for notebook speed (production paths unchanged)
R.N_GRAINS = 24
R.N_PANELS_Y = 4
R.N_PANELS_Z = 4
R.PANEL_SIZE_Y = 150
R.PANEL_SIZE_Z = 150
R.LSD_UM = 7.0e5            # closer detector -> more panels see Au rings
R.N_RINGS = 6
R.TWO_THETA_MAX_DEG = 14.0
R.N_POWDER_PER_RING = 180

# Loss weights (1/sigma per modality so neither dominates) + gauge lambda.
W_POWDER, W_HEDM, LAMBDA_GAUGE = 1.0e4, 10.0, 1.0e6


def build_problem(seed: int = 2026):
    """Build (layout, truth, grains, ring 2theta/d, powder+HEDM obs)."""
    layout = PanelLayout.regular(R.N_PANELS_Y, R.N_PANELS_Z,
                                 R.PANEL_SIZE_Y, R.PANEL_SIZE_Z,
                                 gap_y=R.GAP_Y, gap_z=R.GAP_Z)
    truth = R.sample_truth(layout, seed=seed)
    grain_eulers, grain_pos, grain_lat = R.sample_truth_grains(seed=seed + 1)

    sg = SpaceGroup.from_number(225)                 # Fm-3m (Au)
    lat = Lattice.for_system('cubic', a=R.AU_LATTICE_A)
    _, thetas, _ = hkls_for_forward_model(
        sg, lat, wavelength_A=R.WAVELENGTH_A,
        two_theta_max_deg=R.TWO_THETA_MAX_DEG, expand_equivalents=False)
    ring_tt, _ = torch.unique(2 * thetas * 180.0 / math.pi,
                              return_inverse=True, sorted=True)
    ring_tt = ring_tt.double()[:R.N_RINGS]
    ring_d = R.WAVELENGTH_A / (2.0 * torch.sin(ring_tt * math.pi / 360.0))

    powder_obs = R.generate_powder_observations(layout, truth, ring_tt, seed=seed + 2)
    hedm_obs = R.generate_hedm_observations(
        layout, truth, grain_eulers, grain_pos, grain_lat, seed=seed + 3)
    return dict(layout=layout, truth=truth,
                grain_eulers=grain_eulers, grain_pos=grain_pos, grain_lat=grain_lat,
                ring_tt=ring_tt, ring_d=ring_d,
                powder_obs=powder_obs, hedm_obs=hedm_obs)


def build_spec(prob, *, refine_grains=False, refine_panels=True):
    """Canonical joint spec: geometry + per-panel deltas + grain blocks."""
    truth = prob['truth']; layout = prob['layout']
    spec = mp.ParameterSpec()
    spec.add(mp.Parameter('Lsd', init=truth.Lsd + 50.0,
                          bounds=(truth.Lsd - 5e3, truth.Lsd + 5e3)))
    spec.add(mp.Parameter('BC_y', init=truth.BC_y + 0.3,
                          bounds=(truth.BC_y - 5.0, truth.BC_y + 5.0)))
    spec.add(mp.Parameter('BC_z', init=truth.BC_z - 0.2,
                          bounds=(truth.BC_z - 5.0, truth.BC_z + 5.0)))
    spec.add(mp.Parameter('ty', init=0.0, refined=False))
    spec.add(mp.Parameter('tz', init=0.0, refined=False))
    spec.add(mp.Parameter('Wavelength', init=R.WAVELENGTH_A, refined=False))
    spec.add(mp.Parameter('pxY', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('pxZ', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('RhoD', init=200000.0, refined=False))
    spec.add(mp.Parameter('panel_delta_yz',
                          init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                          bounds=(-3.0, 3.0),
                          prior=mp.GaussianPrior(mean=0.0, std=0.5),
                          refined=refine_panels))
    spec.add(mp.Parameter('panel_delta_theta',
                          init=torch.zeros(layout.n_panels(), dtype=torch.float64),
                          refined=False))
    spec.add(mp.Parameter('grain_euler', init=prob['grain_eulers'],
                          bounds=(-2 * math.pi, 2 * math.pi), refined=refine_grains))
    spec.add(mp.Parameter('grain_pos', init=prob['grain_pos'],
                          bounds=(-1000.0, 1000.0), refined=refine_grains))
    spec.add(mp.Parameter('grain_lattice', init=prob['grain_lat'], refined=False))
    return spec


def make_closures(prob, spec):
    """Return (joint, powder_only, hedm_only) gauge-free residual closures."""
    pf = R.make_powder_residual(prob['powder_obs'], prob['layout'],
                                prob['ring_tt'], ring_d_spacing_A=prob['ring_d'])
    hf = R.make_hedm_residual(prob['hedm_obs'], prob['layout'])

    def joint_fn(u):
        return joint_residual(
            u, powder_residual_fn=pf, hedm_residual_fn=hf, spec=spec,
            weights=JointWeights(w_powder=W_POWDER, w_hedm=W_HEDM,
                                 lambda_gauge=LAMBDA_GAUGE),
            gauge_blocks=[])

    def powder_only(u):   return W_POWDER * pf(u)
    def hedm_only(u):     return W_HEDM * hf(u)
    return joint_fn, powder_only, hedm_only


def seed_unpacked(spec):
    """Dict of every parameter at its init value, as float64 tensors."""
    u = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    for n in u:
        if not isinstance(u[n], torch.Tensor):
            u[n] = torch.tensor(u[n], dtype=torch.float64)
    return u


def covered_panels(prob):
    p = set(prob['powder_obs']['panel_idx'].tolist())
    h = set(prob['hedm_obs']['panel_idx'].tolist())
    return p, h, (p | h)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The canonical parameter names

The joint spec has two families of parameters under fixed names:

**Geometry + detector (shared across all modalities)**

| name | meaning | unit |
|---|---|---|
| `Lsd` | sample-detector distance | µm |
| `BC_y`, `BC_z` | beam centre | px |
| `ty`, `tz` | detector tilts | deg |
| `Wavelength` | X-ray wavelength | Å |
| `pxY`, `pxZ` | pixel pitch | µm |
| `RhoD` | distortion reference radius | µm |
| `panel_delta_yz` | per-panel (δy, δz) shift | px, shape (N_panel, 2) |
| `panel_delta_theta` | per-panel in-plane rotation | rad, shape (N_panel,) |

**HEDM grain nuisance blocks (appended by `build_joint_spec`)**

| name | meaning | unit |
|---|---|---|
| `grain_euler` | per-grain orientation (ZXZ) | rad, shape (N_g, 3) |
| `grain_pos` | per-grain centroid | µm, shape (N_g, 3) |
| `grain_lattice` | per-grain lattice (Voigt a,b,c,α,β,γ) | Å/deg, shape (N_g, 6) |


## Build a powder spec, then extend it with `build_joint_spec`

`build_joint_spec` takes a **powder** `CalibrationSpec` (e.g. from
`midas_calibrate_v2`) and appends the three grain blocks under the
canonical names above.  Default refinement follows the
alternating-driver convention: orientations + positions **on**,
strains **off** (refined in a separate pass).


In [2]:
from midas_joint_ff_calibrate import build_joint_spec

prob = build_problem()
n_g = R.N_GRAINS

# A minimal powder spec (geometry + per-panel deltas).  In production this
# comes from midas_calibrate_v2.compat.from_v1.spec_from_v1_file(...).
powder_spec = build_spec(prob, refine_grains=False, refine_panels=True)
# build_spec already added grain blocks for the notebook helper; for a clean
# demonstration of build_joint_spec we drop them and re-add via the API.
for nm in ('grain_euler', 'grain_pos', 'grain_lattice'):
    if nm in powder_spec.parameters:
        del powder_spec.parameters[nm]

joint_spec = build_joint_spec(
    powder_spec=powder_spec,
    grain_eulers_init=prob['grain_eulers'],
    grain_positions_init=prob['grain_pos'],
    grain_lattices_init=prob['grain_lat'],
    refine_grain_orientation=True,
    refine_grain_position=True,
    refine_grain_strain=False,
)
print(f'joint spec: {len(joint_spec.parameters)} parameter blocks, '
      f'{n_g} grains')


joint spec: 14 parameter blocks, 24 grains


## Inspect the refined / frozen split

`spec.refined_names()` lists the blocks the LM will move.  Note grain
orientation + position are refined; lattice (strain) is frozen.


In [3]:
print('REFINED blocks:')
for n in joint_spec.refined_names():
    p = joint_spec.parameters[n]
    shape = tuple(p.init.shape) if hasattr(p.init, 'shape') and p.init.dim() else '()'
    print(f'  {n:<20s} shape={str(shape):<10s}')

print('\nFROZEN blocks:')
for n, p in joint_spec.parameters.items():
    if n not in joint_spec.refined_names():
        shape = tuple(p.init.shape) if hasattr(p.init, 'shape') and p.init.dim() else '()'
        print(f'  {n:<20s} shape={str(shape):<10s}')


REFINED blocks:
  Lsd                  shape=()        
  BC_y                 shape=()        
  BC_z                 shape=()        
  panel_delta_yz       shape=(16, 2)   
  grain_euler          shape=(24, 3)   
  grain_pos            shape=(24, 3)   

FROZEN blocks:
  ty                   shape=()        
  tz                   shape=()        
  Wavelength           shape=()        
  pxY                  shape=()        
  pxZ                  shape=()        
  RhoD                 shape=()        
  panel_delta_theta    shape=(16,)     
  grain_lattice        shape=(24, 6)   


## Pack / unpack round-trip

The spec packs to a flat refined vector for the LM, and unpacks back
to a name->tensor dict for the residual closures.


In [4]:
x, info = mp.pack_spec(joint_spec)
unpacked = mp.unpack_spec(x, info, joint_spec)
print(f'packed refined vector length: {x.numel()}')
print(f'unpacked keys: {sorted(unpacked.keys())}')
print(f"grain_euler shape: {tuple(unpacked['grain_euler'].shape)}")
print(f"panel_delta_yz shape: {tuple(unpacked['panel_delta_yz'].shape)}")


packed refined vector length: 345
unpacked keys: ['BC_y', 'BC_z', 'Lsd', 'RhoD', 'Wavelength', 'grain_euler', 'grain_lattice', 'grain_pos', 'panel_delta_theta', 'panel_delta_yz', 'pxY', 'pxZ', 'ty', 'tz']
grain_euler shape: (24, 3)
panel_delta_yz shape: (16, 2)


## Next

* **01** — the recommended `AlternatingDriver` on synthetic Pilatus.
* **02** — `FullJointDriver` + Laplace covariance (per-panel σ).
* **03** — the Fisher block-rank diagnostic (paper headline).
* **04** — the (Lsd, λ) gauge story.
